In [ ]:
!pip install pyspark
!pip install findspark

# Import SparkSession
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Check Spark Session Information
spark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


RuntimeError: ignored

In [ ]:
# (1) Import the required Python dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import findspark
findspark.init()
from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors
from pyspark.ml.linalg import DenseVector
from pyspark.mllib.linalg.distributed import RowMatrix

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

lines = spark.read.text("/content/sample_movielens_ratings.txt").rdd
parts = lines.map(lambda row: row.value.split("::"))
ratingsRDD = parts.map(lambda p: Row(userId=int(p[0]), movieId=int(p[1]),
                                     rating=float(p[2]), timestamp=int(p[3])))
ratings = spark.createDataFrame(ratingsRDD)
(training, test) = ratings.randomSplit([0.8, 0.2])

# Build the recommendation model using ALS on the training data
# Note we set cold start strategy to 'drop' to ensure we don't get NaN evaluation metrics
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(training)

# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

# Generate top 10 movie recommendations for each user
userRecs = model.recommendForAllUsers(10)
# Generate top 10 user recommendations for each movie
movieRecs = model.recommendForAllItems(10)

# Generate top 10 movie recommendations for a specified set of users
users = ratings.select(als.getUserCol()).distinct().limit(3)
userSubsetRecs = model.recommendForUserSubset(users, 10)
# Generate top 10 user recommendations for a specified set of movies
movies = ratings.select(als.getItemCol()).distinct().limit(3)
movieSubSetRecs = model.recommendForItemSubset(movies, 10)

# $example off$
userRecs.show()
movieRecs.show()
userSubsetRecs.show()
movieSubSetRecs.show()


Root-mean-square error = 1.7399622397998498
+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|    20|[{68, 5.7477283},...|
|    10|[{4, 3.966461}, {...|
|     0|[{92, 3.9999123},...|
|     1|[{68, 4.0175295},...|
|    21|[{53, 5.157362}, ...|
|    11|[{30, 5.0815372},...|
|    12|[{49, 5.3215427},...|
|    22|[{46, 5.8628254},...|
|     2|[{8, 5.032946}, {...|
|    13|[{85, 4.471457}, ...|
|     3|[{32, 6.136252}, ...|
|    23|[{80, 5.5843596},...|
|     4|[{76, 4.4393196},...|
|    24|[{8, 5.2096453}, ...|
|    14|[{52, 4.9376183},...|
|     5|[{80, 5.138089}, ...|
|    15|[{46, 4.8454776},...|
|    25|[{46, 4.4676137},...|
|    26|[{46, 7.2310376},...|
|     6|[{52, 4.7155437},...|
+------+--------------------+
only showing top 20 rows

+-------+--------------------+
|movieId|     recommendations|
+-------+--------------------+
|     20|[{17, 4.7682796},...|
|     40|[{10, 3.9430225},...|
|     10|[{28, 4.9843636},...|
|     50|[{23, 3.8127913